<div align="right"><a href="S06-Sine-Generator.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L06-Sinus-Generator.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L05 Frequenzgang und Wobbelmessung](./L05-Frequenzgang-und-Wobbelmessung.ipynb) | [Aufgabe 06](./A06-Sinus-Generator.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L07 Faltungsmatrix](./L07-Faltungsmatrix.ipynb)

</div>

---


# Sinusgenerator

In dieser Aufgabe soll ein Sinusgenerator für eine feste Frequenz realisiert werden, der **ohne Tabellen oder Polynomapproximationen** arbeitet. Die Grundlage dieses Generators ist ein **System 2. Ordnung**, das nach einmaliger Anregung die Werte eines Sinussignals liefert.

Für den Syntheseprozess soll folgende Z-Transformationskorrespondenz verwendet werden:

$$
h(k) = \sin!\big(\Omega_0 k\big)
$$

$$
H(z) = Z{h(k)} = \frac{z ,\sin(\Omega_0)}{z^2 - 2 z \cdot \cos(\Omega_0) + 1}
$$

Diese Korrespondenz kann als **Impulsantwort** $h(k)$ eines Systems $H(z) = Y(z)/X(z)$ interpretiert werden.

---

### a)

Zeichnen Sie ein **Blockschaltbild** des resultierenden Sinusgenerators.

**Hinweis:** Formen Sie die obige Gleichung zunächst in eine Darstellung um, die nur Verzögerungen enthält (Terme der Form $z^{-d}$).

---

### b)

Realisieren Sie das System, indem Sie seine **Koeffizientenvektoren** bestimmen:

```text
b = [b_0, b_1, b_2]  
a = [a_0, a_1, a_2]
```

für die Frequenz $\Omega_0 = 1/32$.

Zeichnen Sie das **Nullstellen-Pol-Plot** mit Python (z. B. mit `matplotlib` oder `scipy.signal`).

Bestimmen Sie die **Impulsantwort** des Systems mit Python (z. B. `scipy.signal.dimpulse` oder `lfilter`). Vergleichen Sie das Ergebnis mit einem gesampelten Sinussignal der Frequenz $\Omega_0 = 1/32$.

---

### c)

Welche **Auswirkung** hat der Vorwärtsweg dieses Systems?

Was passiert, wenn das Verzögerungsglied und das Multiplikationselement im Vorwärtsweg entfernt werden?

Überprüfen Sie Ihre Annahme, indem Sie die Koeffizientenvektoren `a` und `b` entsprechend ändern und die Impulsantwort mit Python neu berechnen.

---

### d)

Untersuchen Sie nun die Effekte der **Quantisierung der Systemkoeffizienten**.

Ersetzen Sie dazu den Ausdruck $2 \cdot \cos(\Omega_0)$ im Koeffizientenvektor `a` durch $2.1 \cdot \cos(\Omega_0)$.

Zeigen Sie das **Nullstellen-Pol-Plot** und die Impulsantwort des Systems mit Python.

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import dlti, dimpulse, tf2zpk

# Normalized dimensionless angular frequency
Omega0 = 2 * np.pi * 1 / 32

# Sampled sine signal
k = np.arange(0, 100)
x = np.sin(Omega0 * k)

# ##### Subtask b: Sine Generator #####
b = [0, np.sin(Omega0), 0]
a = [1, -2 * np.cos(Omega0), 1]

zeros, poles, gain = tf2zpk(b, a)

# z-Plane Plot
plt.figure()
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)

# Draw the unit circle
theta = np.linspace(0, 2*np.pi, 500)
plt.plot(np.cos(theta), np.sin(theta), 'k--', label='Unit Circle')

# Plot zeros and poles
plt.scatter(np.real(zeros), np.imag(zeros), marker='o', facecolors='none', edgecolors='b', label='Zeros')
plt.scatter(np.real(poles), np.imag(poles), marker='x', color='r', label='Poles')

plt.gca().set_aspect('equal')  # Ensure circle is not distorted
plt.grid(True)
plt.xlabel('Real Part')
plt.ylabel('Imaginary Part')
plt.title('z-Plane')
plt.show()


# Impulse response
system = dlti(b, a)
t, h = dimpulse(system, n=100)
h = np.squeeze(h)

plt.figure()
plt.stem(t, h)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title('Impulse Response')
plt.plot(k, x, 'r-', label='Sampled Sine Signal')
plt.legend()
plt.show()

# ##### Subtask c: Forward Branch #####
b = [1, 0, 0]
a = [1, -2 * np.cos(Omega0), 1]

# Graphical output
system = dlti(b, a)
t, h = dimpulse(system, n=100)
h = np.squeeze(h)

plt.figure()
plt.stem(t, h)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title('Impulse Response')
plt.show()

# ##### Subtask d: Instability #####
b = [0, np.sin(Omega0), 0]
a = [1, -2.1 * np.cos(Omega0), 1]

# Graphical output
# Poles and zeros in the z-plane
plt.figure()
zeros, poles, gain = tf2zpk(b, a)

theta = np.linspace(0, 2*np.pi, 500)
plt.plot(np.cos(theta), np.sin(theta), 'k--', label='Unit Circle')

plt.scatter(np.real(zeros), np.imag(zeros), marker='o', label='Zeros')
plt.scatter(np.real(poles), np.imag(poles), marker='x', label='Poles')

plt.gca().set_aspect('equal')  # Ensure circle is not distorted
plt.xlabel('Real Part')
plt.ylabel('Imaginary Part')
plt.title('z-Plane')
plt.grid()
plt.show()

# Impulse response
system = dlti(b, a)
t, h = dimpulse(system, n=100)
h = np.squeeze(h)

plt.figure()
plt.stem(t, h)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title('Impulse Response')
plt.show()
# END SOLUTION

## Lösung: Sinusgenerator

#### a)

$$
H(z) = \frac{Y(z)}{X(z)} = \frac{z^{-1} \sin(\Omega_0)}{1 - 2 z^{-1} \cos(\Omega_0) + z^{-2}}
$$

Nach der Transformation von $H(z)$ lässt sich folgendes **Blockschaltbild** zeichnen:

<img src="./pngs/singen.png" width="250">

---

#### b)

Die **Koeffizientenvektoren** des Systems sind:

$$
\mathbf{b} = {0, \sin(\Omega_0), 0}
$$

und

$$
\mathbf{a} = {1, -2 \cos(\Omega_0), 1}.
$$

Das Nullstellen-Pol-Plot zeigt, dass die **Pole auf dem Einheitskreis** liegen und das System somit am Rande der Stabilität arbeitet. Die Position der Pole bestimmt die Frequenz des Sinusgenerators.

<img src="./pngs/ldsv2a21_b1.png" width="250">
<img src="./pngs/ldsv2a21_b2.png" width="250">

Ein Vergleich der Impulsantwort des Generatorsystems mit einem gesampelten Sinussignal zeigt keine Unterschiede.

---

#### c)

Wenn das Verzögerungsglied und das Multiplikationselement im Vorwärtsweg entfernt werden, ergibt sich folgende Impulsantwort:

<img src="./pngs/ldsv2a21_c1.png" width="250">

Ein **Phasenversatz** sowie eine **Amplitudenzunahme** sind deutlich sichtbar.

---

#### d)

Eine Erhöhung des Rückkopplungsfaktors führt zu **Instabilität** des Systems. Ein Pol liegt außerhalb des Einheitskreises, und die Impulsantwort wächst unbegrenzt.

<img src="./pngs/ldsv2a21_d1.png" width="250">
<img src="./pngs/ldsv2a21_d2.png" width="250">